In [ ]:
import os
os.system("python3 split_pred_by_template.py --pred /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/Protenix/2000-0.1T-dimer/1czy_sample_1/ligandmpnn_seed42/pred.json --template ../template.json --outdir ./ --seeds 42 43 44 --template-needed-chains 1")

In [1]:
import os
os.system("python pipeline_metrics.py --pdb-list /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed/PDB.list \
           --ref-pdb-base /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed \
           --output-base /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/predict/AlphaFold3/PepSet_AF3_pass-2k_0.1/outputs/ligandmpnn_seed42 \
           --peptide-chain B")

匹配到设计目录: 1100
二级sample目录总数: 16500
成功写入行数: 16500
CSV saved to: /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/predict/AlphaFold3/PepSet_AF3_pass-2k_0.1/outputs/ligandmpnn_seed42/success_rates_by_parent_complex.csv

不同scRMSD阈值下的平均成功率(%)：
 scRMSD_cutoff  mean_success_rate
           1.0          36.727273
           1.5          45.636364
           2.0          50.545455
           2.5          53.454545


0

In [1]:
import os
os.system("python pipeline_metrics_parallel.py \
            --pdb-list /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed/PDB.list \
            --ref-pdb-base /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed \
            --output-base ./PepSet_AF3_pass-2k_0.1/outputs/ligandmpnn_seed43 \
            --peptide-chain B \
            --num-workers 5 \
            --progress-every 10")

可处理parent_complex数量: 110
并行进程数: 5
进度: 10/110 parent_complex 完成
进度: 20/110 parent_complex 完成
进度: 30/110 parent_complex 完成
进度: 40/110 parent_complex 完成
进度: 50/110 parent_complex 完成
进度: 60/110 parent_complex 完成
进度: 70/110 parent_complex 完成
进度: 80/110 parent_complex 完成
进度: 90/110 parent_complex 完成
进度: 100/110 parent_complex 完成
进度: 110/110 parent_complex 完成
匹配到设计目录: 1100
二级sample目录总数: 16500
成功写入行数: 16500
CSV saved to: PepSet_AF3_pass-2k_0.1/outputs/ligandmpnn_seed43/success_rates_by_parent_complex.csv

不同scRMSD阈值下的平均成功率(%)：
 scRMSD_cutoff  mean_success_rate
           1.0          35.090909
           1.5          45.000000
           2.0          49.727273
           2.5          53.363636


0

In [2]:
import os
os.system("python pipeline_metrics_parallel.py \
            --pdb-list /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed/PDB.list \
            --ref-pdb-base /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed \
            --output-base ./PepSet_AF3_pass-2k_0.1/outputs/ligandmpnn_seed44 \
            --peptide-chain B \
            --num-workers 5 \
            --progress-every 10")

可处理parent_complex数量: 110
并行进程数: 5
进度: 10/110 parent_complex 完成
进度: 20/110 parent_complex 完成
进度: 30/110 parent_complex 完成
进度: 40/110 parent_complex 完成
进度: 50/110 parent_complex 完成
进度: 60/110 parent_complex 完成
进度: 70/110 parent_complex 完成
进度: 80/110 parent_complex 完成
进度: 90/110 parent_complex 完成
进度: 100/110 parent_complex 完成
进度: 110/110 parent_complex 完成
匹配到设计目录: 1100
二级sample目录总数: 16500
成功写入行数: 16500
CSV saved to: PepSet_AF3_pass-2k_0.1/outputs/ligandmpnn_seed44/success_rates_by_parent_complex.csv

不同scRMSD阈值下的平均成功率(%)：
 scRMSD_cutoff  mean_success_rate
           1.0          36.363636
           1.5          45.909091
           2.0          51.272727
           2.5          53.818182


0

# 对PepSet结构预测后的通过的结果进行三次LigandMPNN设计，并对设计结果利用AF3进行筛选与预测。计算相较于设计前的预测结构的scRMSD

In [12]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from Bio.PDB import PDBParser, FastMMCIFParser, Superimposer

PDB_LIST_PATH = Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/datasets/PepSet-passed-dimer_processed/PDB.list")
REF_PDB_BASE = Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/pred/dimer_filtered")
OUTPUTS_BASE = Path("./outputs")

LIGANDMPNN_SEEDS = ("lig42",)
ALLOWED_PROTENIX_SEEDS = {42, 43, 44}
PEPTIDE_CHAIN_ID = "B"
BACKBONE_ATOMS = {"N", "CA", "C", "O"}


def read_pdb_list(pdb_list_path: Path) -> list[str]:
    with pdb_list_path.open("r") as f:
        return [line.strip() for line in f if line.strip()]


def parse_seed_sample(sample_name: str) -> tuple[int, int] | None:
    match = re.fullmatch(r"seed-(\d+)_sample-(\d+)", sample_name)
    if match is None:
        return None
    seed = int(match.group(1))
    sample_id = int(match.group(2))
    return seed, sample_id


def get_backbone_atoms(chain):
    return [atom for atom in chain.get_atoms() if atom.get_id() in BACKBONE_ATOMS]


def get_first_non_target_chain(model, target_chain_id: str):
    for chain in model:
        if chain.id != target_chain_id:
            return chain
    return None


def compute_sc_rmsd(ref_chain, pred_chain, align_ref_chain, align_pred_chain) -> float | None:
    align_ref_atoms = get_backbone_atoms(align_ref_chain)
    align_pred_atoms = get_backbone_atoms(align_pred_chain)
    if len(align_ref_atoms) == 0 or len(align_pred_atoms) == 0:
        return None
    if len(align_ref_atoms) != len(align_pred_atoms):
        return None

    sup = Superimposer()
    try:
        sup.set_atoms(align_ref_atoms, align_pred_atoms)
    except Exception:
        return None
    rot, tran = sup.rotran

    ref_atoms = get_backbone_atoms(ref_chain)
    pred_atoms = get_backbone_atoms(pred_chain)
    if len(ref_atoms) == 0 or len(pred_atoms) == 0:
        return None
    if len(ref_atoms) != len(pred_atoms):
        return None

    ref_coords = np.array([atom.get_coord() for atom in ref_atoms], dtype=float)
    pred_coords = np.array([atom.get_coord() for atom in pred_atoms], dtype=float)
    pred_coords_aligned = pred_coords @ rot + tran

    diffs = ref_coords - pred_coords_aligned
    return round(float(np.sqrt(np.mean(np.sum(diffs**2, axis=1)))), 4)


def process_single_sample(
    pdb: str,
    sample_prefix: str,
    sample_dir: Path,
    structure_ref,
    chain_ref_pep,
    mmcifparser: FastMMCIFParser,
    ) -> tuple[tuple | None, str | None]:
    sample_name = sample_dir.name
    parsed = parse_seed_sample(sample_name)
    if parsed is None:
        return None, "bad_sample_dir_name"
    seed, sample_id = parsed
    if seed not in ALLOWED_PROTENIX_SEEDS:
        return None, "seed_not_allowed"

    conf_json_path = sample_dir / f"{sample_prefix}_{sample_name}_confidences.json"
    summary_json_path = sample_dir / f"{sample_prefix}_{sample_name}_summary_confidences.json"
    cif_path = sample_dir / f"{sample_prefix}_{sample_name}_model.cif"
    if not (conf_json_path.exists() and summary_json_path.exists() and cif_path.exists()):
        return None, "missing_required_files"

    with conf_json_path.open("r") as f_json:
        conf_data = json.load(f_json)
    atom_chain_ids = conf_data.get("atom_chain_ids", [])
    atom_plddts = conf_data.get("atom_plddts", [])
    if len(atom_chain_ids) == 0 or len(atom_chain_ids) != len(atom_plddts):
        return None, "invalid_confidences_json"

    b_chain_plddt = [
        float(plddt)
        for chain_id, plddt in zip(atom_chain_ids, atom_plddts)
        if chain_id == PEPTIDE_CHAIN_ID
    ]
    if len(b_chain_plddt) == 0:
        return None, "missing_chain_B_plddt"
    pep_plddt = round(float(np.mean(b_chain_plddt)), 4)

    with summary_json_path.open("r") as f_json:
        summary_data = json.load(f_json)
    if "iptm" not in summary_data or "ranking_score" not in summary_data:
        return None, "invalid_summary_json"
    iptm = round(float(summary_data["iptm"]), 4)
    ranking_score = round(float(summary_data["ranking_score"]), 4)

    structure_pred = mmcifparser.get_structure("pred", str(cif_path))
    try:
        chain_pred_pep = structure_pred[0][PEPTIDE_CHAIN_ID]
    except KeyError:
        return None, "missing_chain_B_in_pred"

    align_chain_ref = get_first_non_target_chain(structure_ref[0], PEPTIDE_CHAIN_ID)
    align_chain_pred = get_first_non_target_chain(structure_pred[0], PEPTIDE_CHAIN_ID)
    if align_chain_ref is None or align_chain_pred is None:
        return None, "missing_align_chain"

    sc_rmsd = compute_sc_rmsd(chain_ref_pep, chain_pred_pep, align_chain_ref, align_chain_pred)
    if sc_rmsd is None:
        return None, "rmsd_alignment_failed"

    return (
        sample_prefix,
        seed,
        sample_id,
        pep_plddt,
        iptm,
        ranking_score,
        sc_rmsd,
    ), None


def collect_metrics_for_ligand_seed(
    ligand_seed: str,
    pdbs: list[str],
    pdbparser: PDBParser,
    mmcifparser: FastMMCIFParser,
    ) -> pd.DataFrame:
    predicted_pdb_dir = OUTPUTS_BASE / ligand_seed
    if not predicted_pdb_dir.is_dir():
        print(f"[{ligand_seed}] 目录不存在: {predicted_pdb_dir}")
        return pd.DataFrame(columns=["complex", "seed", "id", "pep_plddt", "iptm", "ranking_score", "scRMSD"] )

    rows = []
    matched_pdb_dirs = 0
    total_sample_dirs = 0
    skip_reasons = {}

    for pdb in pdbs:
        ref_pdb_path = REF_PDB_BASE / f"{pdb}.pdb"
        if not ref_pdb_path.exists():
            raise FileNotFoundError(f"Reference PDB file not found: {ref_pdb_path}")

        structure_ref = pdbparser.get_structure("ref", str(ref_pdb_path))
        try:
            chain_ref_pep = structure_ref[0][PEPTIDE_CHAIN_ID]
        except KeyError:
            skip_reasons["missing_chain_B_in_ref"] = skip_reasons.get("missing_chain_B_in_ref", 0) + 1
            continue

        pdb_dir_pattern = re.compile(rf"^{re.escape(pdb)}_\d+$")
        pdb_dirs = [
            d for d in predicted_pdb_dir.iterdir()
            if d.is_dir() and pdb_dir_pattern.fullmatch(d.name)
        ]
        matched_pdb_dirs += len(pdb_dirs)

        for pdb_dir in sorted(pdb_dirs):
            sample_dirs = [d for d in pdb_dir.iterdir() if d.is_dir()]
            total_sample_dirs += len(sample_dirs)
            for sample_dir in sorted(sample_dirs):
                row, reason = process_single_sample(
                    pdb=pdb,
                    sample_prefix=pdb_dir.name,
                    sample_dir=sample_dir,
                    structure_ref=structure_ref,
                    chain_ref_pep=chain_ref_pep,
                    mmcifparser=mmcifparser,
                )
                if row is not None:
                    rows.append(row)
                else:
                    skip_reasons[reason] = skip_reasons.get(reason, 0) + 1

    result = pd.DataFrame(
        rows,
        columns=["complex", "seed", "id", "pep_plddt", "iptm", "ranking_score", "scRMSD"],
    )

    result.to_csv(predicted_pdb_dir / "metrics_summary-seed42_43_44.csv", index=False)
    result_filtered = result[result["scRMSD"] <= 2.5].reset_index(drop=True)
    result_filtered.to_csv(predicted_pdb_dir / "metrics_filtered_scRMSD_le25-seed42_43_44.csv", index=False)

    print(f"[{ligand_seed}] 匹配到一级目录: {matched_pdb_dirs}")
    print(f"[{ligand_seed}] 二级sample目录总数: {total_sample_dirs}")
    print(f"[{ligand_seed}] 成功写入行数: {len(result)}")
    if skip_reasons:
        print(f"[{ligand_seed}] 跳过原因统计: {skip_reasons}")

    return result_filtered


pdbs = read_pdb_list(PDB_LIST_PATH)
pdbparser = PDBParser(QUIET=True)
mmcifparser = FastMMCIFParser(QUIET=True)

result_filtered_by_seed = {}
for ligandmpnn_seed in LIGANDMPNN_SEEDS:
    result_filtered_by_seed[ligandmpnn_seed] = collect_metrics_for_ligand_seed(
        ligand_seed=ligandmpnn_seed,
        pdbs=pdbs,
        pdbparser=pdbparser,
        mmcifparser=mmcifparser,
    )

result_filtered_by_seed

[lig42] 匹配到一级目录: 906
[lig42] 二级sample目录总数: 13590
[lig42] 成功写入行数: 13590


{'lig42':               complex  seed  id  pep_plddt  iptm  ranking_score  scRMSD
 0     1f47_sample_2_1    42   0    82.6182  0.83           0.89  1.0739
 1     1f47_sample_2_1    42   1    83.3186  0.83           0.90  0.9572
 2     1f47_sample_2_1    42   2    82.6323  0.82           0.89  0.8569
 3     1f47_sample_2_1    42   3    83.2493  0.84           0.90  0.7650
 4     1f47_sample_2_1    42   4    83.8742  0.85           0.91  0.8334
 ...               ...   ...  ..        ...   ...            ...     ...
 8204  6h7b_sample_1_9    44   0    95.5043  0.92           0.99  0.9641
 8205  6h7b_sample_1_9    44   1    95.2763  0.92           0.98  0.8729
 8206  6h7b_sample_1_9    44   2    95.4282  0.92           0.98  0.8701
 8207  6h7b_sample_1_9    44   3    95.2651  0.91           0.98  0.8521
 8208  6h7b_sample_1_9    44   4    95.2925  0.91           0.98  1.0585
 
 [8209 rows x 7 columns]}

In [13]:
import re
from pathlib import Path

import pandas as pd

cutoffs = [1.0, 1.5, 2.0, 2.5]
pep_plddt_cutoff = 70
iptm_cutoff = 0.7

outputs_base = Path("./outputs")
ligand_seeds = ["lig42"]


def get_parent_complex(complex_name: str) -> str:
    return re.sub(r"_\d+$", "", complex_name)


all_rows = []

for ligand_seed in ligand_seeds:
    summary_csv = outputs_base / ligand_seed / "metrics_summary-seed42_43_44.csv"
    if not summary_csv.exists():
        print(f"[{ligand_seed}] 未找到文件: {summary_csv}")
        continue

    df = pd.read_csv(summary_csv)
    if df.empty:
        print(f"[{ligand_seed}] 指标为空: {summary_csv}")
        continue

    df = df.copy()
    df["parent_complex"] = df["complex"].astype(str).map(get_parent_complex)

    design_counts = (
        df[["parent_complex", "complex"]]
        .drop_duplicates()
        .groupby("parent_complex", as_index=False)
        .agg(total_designs=("complex", "nunique"))
    )

    for cutoff in cutoffs:
        passed = df[
            (df["pep_plddt"] >= pep_plddt_cutoff)
            & (df["iptm"] >= iptm_cutoff)
            & (df["scRMSD"] <= cutoff)
        ]

        success_counts = (
            passed[["parent_complex", "complex"]]
            .drop_duplicates()
            .groupby("parent_complex", as_index=False)
            .agg(successful_designs=("complex", "nunique"))
        )

        stat = design_counts.merge(success_counts, on="parent_complex", how="left")
        stat["successful_designs"] = stat["successful_designs"].fillna(0).astype(int)
        stat["success_rate"] = (stat["successful_designs"] / stat["total_designs"] * 100).round(2)

        stat["ligand_seed"] = ligand_seed
        stat["pep_plddt_cutoff"] = pep_plddt_cutoff
        stat["iptm_cutoff"] = iptm_cutoff
        stat["scRMSD_cutoff"] = cutoff

        all_rows.append(
            stat[[
                "ligand_seed",
                "parent_complex",
                "total_designs",
                "successful_designs",
                "success_rate",
                "pep_plddt_cutoff",
                "iptm_cutoff",
                "scRMSD_cutoff",
            ]]
        )

if not all_rows:
    print("没有可统计的数据")
else:
    success_rate_df = pd.concat(all_rows, ignore_index=True).sort_values(
        ["ligand_seed", "parent_complex", "scRMSD_cutoff"]
    )

    output_path = outputs_base / "success_rates_by_parent_complex.csv"
    success_rate_df.to_csv(output_path, index=False)
    print(f"CSV saved to: {output_path}")

    summary = (
        success_rate_df.groupby(["ligand_seed", "scRMSD_cutoff"], as_index=False)["success_rate"]
        .mean()
        .rename(columns={"success_rate": "mean_success_rate"})
    )
    print("\n不同scRMSD阈值下的平均成功率(%)：")
    print(summary.to_string(index=False))

    success_rate_df.head(20)

CSV saved to: outputs/success_rates_by_parent_complex.csv

不同scRMSD阈值下的平均成功率(%)：
ligand_seed  scRMSD_cutoff  mean_success_rate
      lig42            1.0          36.153846
      lig42            1.5          51.016484
      lig42            2.0          55.961538
      lig42            2.5          60.686813


# 检查每一个结构预测的15个结构的序列是否json文件的序列保持一致

In [14]:
# 检查每一个结构预测后的序列是否与对应json中的多肽(B链)序列一致
import json
from pathlib import Path

import pandas as pd
from Bio.PDB import FastMMCIFParser
from Bio.SeqUtils import seq1

outputs_base = Path("./outputs")
inputs_base = Path("./inputs")

# 当前先检查lig42，可在后续补充lig43/lig44
seed_mapping = {
    "lig42": "ligandmpnn_seed42",
}


def get_expected_chain_b_sequence(json_path: Path) -> str | None:
    with json_path.open("r") as f:
        data = json.load(f)

    jobs = data if isinstance(data, list) else [data]
    if len(jobs) == 0:
        return None

    # 这里每个json通常对应一个job，优先取第一个
    job = jobs[0]
    sequences = job.get("sequences", [])

    for item in sequences:
        if not isinstance(item, dict):
            continue
        for key in ("protein", "proteinChain"):
            chain_info = item.get(key, {})
            if not isinstance(chain_info, dict):
                continue
            if str(chain_info.get("id", "")) == "B":
                seq = chain_info.get("sequence")
                if isinstance(seq, str) and seq:
                    return seq
    return None


def extract_chain_b_sequence_from_cif(cif_path: Path, parser: FastMMCIFParser) -> str | None:
    structure = parser.get_structure("pred", str(cif_path))
    try:
        chain_b = structure[0]["B"]
    except KeyError:
        return None

    aa_seq = []
    for residue in chain_b.get_residues():
        if residue.get_id()[0] != " ":
            continue
        aa_seq.append(seq1(residue.get_resname(), custom_map={"MSE": "M"}, undef_code="X"))
    return "".join(aa_seq)


rows = []
mmcifparser = FastMMCIFParser(QUIET=True)

for lig_seed, input_seed_dir in seed_mapping.items():
    lig_output_dir = outputs_base / lig_seed
    lig_input_dir = inputs_base / input_seed_dir

    if not lig_output_dir.is_dir():
        print(f"[{lig_seed}] 输出目录不存在: {lig_output_dir}")
        continue
    if not lig_input_dir.is_dir():
        print(f"[{lig_seed}] 输入目录不存在: {lig_input_dir}")
        continue

    design_dirs = sorted([d for d in lig_output_dir.iterdir() if d.is_dir()])
    for design_dir in design_dirs:
        design_name = design_dir.name
        json_path = lig_input_dir / f"{design_name}.json"

        if not json_path.exists():
            rows.append({
                "ligand_seed": lig_seed,
                "design": design_name,
                "seed": None,
                "sample": None,
                "status": "missing_json",
                "expected_seq": None,
                "pred_seq": None,
            })
            continue

        expected_seq = get_expected_chain_b_sequence(json_path)
        if not expected_seq:
            rows.append({
                "ligand_seed": lig_seed,
                "design": design_name,
                "seed": None,
                "sample": None,
                "status": "invalid_json_or_missing_chain_B",
                "expected_seq": None,
                "pred_seq": None,
            })
            continue

        pred_dirs = sorted([
            d for d in design_dir.iterdir()
            if d.is_dir() and d.name.startswith("seed-") and "_sample-" in d.name
        ])

        for pred_dir in pred_dirs:
            try:
                seed_part, sample_part = pred_dir.name.split("_sample-")
                seed_id = int(seed_part.replace("seed-", ""))
                sample_id = int(sample_part)
            except Exception:
                rows.append({
                    "ligand_seed": lig_seed,
                    "design": design_name,
                    "seed": None,
                    "sample": None,
                    "status": "bad_pred_dir_name",
                    "expected_seq": expected_seq,
                    "pred_seq": None,
                })
                continue

            cif_path = pred_dir / f"{design_name}_{pred_dir.name}_model.cif"
            if not cif_path.exists():
                rows.append({
                    "ligand_seed": lig_seed,
                    "design": design_name,
                    "seed": seed_id,
                    "sample": sample_id,
                    "status": "missing_cif",
                    "expected_seq": expected_seq,
                    "pred_seq": None,
                })
                continue

            pred_seq = extract_chain_b_sequence_from_cif(cif_path, mmcifparser)
            if pred_seq is None:
                rows.append({
                    "ligand_seed": lig_seed,
                    "design": design_name,
                    "seed": seed_id,
                    "sample": sample_id,
                    "status": "missing_chain_B_in_cif",
                    "expected_seq": expected_seq,
                    "pred_seq": None,
                })
                continue

            status = "match" if pred_seq == expected_seq else "mismatch"
            rows.append({
                "ligand_seed": lig_seed,
                "design": design_name,
                "seed": seed_id,
                "sample": sample_id,
                "status": status,
                "expected_seq": expected_seq,
                "pred_seq": pred_seq,
            })

check_df = pd.DataFrame(rows)
if check_df.empty:
    print("没有可检查的数据")
else:
    output_csv = outputs_base / "sequence_consistency_check.csv"
    check_df.to_csv(output_csv, index=False)
    print(f"CSV saved to: {output_csv}")

    status_summary = check_df["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="count")
    print("\n状态统计：")
    print(status_summary.to_string(index=False))

    mismatch_df = check_df[check_df["status"] == "mismatch"]
    print(f"\nMismatch条目数: {len(mismatch_df)}")
    mismatch_df.head(20)

CSV saved to: outputs/sequence_consistency_check.csv

状态统计：
status  count
 match  13590

Mismatch条目数: 0


In [ ]:
# 